# 28. E5 입력 길이 보정과 하이브리드 검색 비교

**목적:** 27번에서 일부 청크가 E5의 512-token 입력 한도를 넘어 뒤쪽이 잘린 문제를 먼저 해결하고, 같은 새 질문셋에서 `문자 n-gram + E5`와 `Kiwi 형태소 BM25 + E5`를 비교합니다.

**공정한 비교 조건:** 같은 삼성전자·리노공업 2024 사업보고서, 같은 E5 토크나이저 기준 청크, 같은 E5 인덱스·질문, 같은 RRF 깊이 20·k=60을 사용합니다. 다른 것은 어휘 검색 arm뿐입니다. 질문 10개는 27번 질문과 문장이 겹치지 않으며, 근거가 확인된 기존 보고서에서 작성한 **개발용 질문**입니다. 새로운 회사·보고서에서 독립적으로 만든 holdout은 아닙니다.

**판단:** Hit@3, Hit@5, MRR@5와 질문별 근거 순위·실패를 함께 봅니다. 결측 근거나 E5 초과 청크가 있으면 검색 비교를 중단합니다. 결과가 엇갈리면 바로 채택하지 않습니다. 마지막 5절은 같은 청크·질문에서 `text-embedding-3-small`을 추가해 **임베딩 모델만** 비교합니다. 이 실험은 LLM 답변·reranker·구조화 DB 검색을 다루지 않습니다.

## 1. 환경과 고정 질문

`skn25` 커널에서 위에서 아래로 실행합니다. OpenDART 원문을 다시 받으므로 `.env`의 유효한 `OPENDART_API_KEY`가 필요합니다. 키 값·요청 URL·원문 전체는 출력하지 않습니다. 모델은 CPU에서 실행해 GPU에 올려둔 Qwen을 건드리지 않습니다.

In [1]:
from io import BytesIO
from pathlib import Path
from time import perf_counter
import os, re, sqlite3, zipfile

from bs4 import BeautifulSoup, Comment, NavigableString, Tag
from dotenv import load_dotenv
from IPython.display import display
from kiwipiepy import Kiwi
from llama_index.core import Document
from llama_index.core.node_parser import SentenceSplitter
from sklearn.feature_extraction.text import HashingVectorizer
from transformers import AutoModel, AutoTokenizer
import numpy as np
import pandas as pd
import requests
import torch
import torch.nn.functional as F

PROJECT_ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (p / '.env').exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('프로젝트 최상위 .env를 찾지 못했습니다.')
load_dotenv(PROJECT_ROOT / '.env')
API_KEY = os.getenv('OPENDART_API_KEY')
if not API_KEY:
    raise RuntimeError('.env에 OPENDART_API_KEY가 필요합니다.')
print('환경 확인 완료 | skn25 커널인지 확인하세요. API 키 값은 출력하지 않습니다.')

/home/sms/anaconda3/envs/skn25/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


환경 확인 완료 | skn25 커널인지 확인하세요. API 키 값은 출력하지 않습니다.


In [2]:
REPORTS = {
    '삼성전자': {'rcept_no': '20250311001085', 'xml_name': '20250311001085.xml'},
    '리노공업': {'rcept_no': '20250318000229', 'xml_name': '20250318000229.xml'},
}
CASES = [
    {'id': 'S01', 'company': '삼성전자', 'question': '삼성전자가 주식을 일반에 공개한 시점은 언제인가?', 'quote': '1975년 6월 11일 기업공개를 실시하였습니다.'},
    {'id': 'S02', 'company': '삼성전자', 'question': '삼성전자공업에서 지금의 회사명으로 바뀐 날짜는?', 'quote': '1984년 2월 28일 정기주주총회 결의에 의거하여 상호를'},
    {'id': 'S03', 'company': '삼성전자', 'question': '삼성전자 본사의 소재지는 어디인가?', 'quote': '경기도 수원시 영통구 삼성로 129(매탄동)'},
    {'id': 'S04', 'company': '삼성전자', 'question': '삼성전자 반도체 사업 부문은 어떤 제품을 주로 만드는가?', 'quote': 'DS 부문 | DRAM, NAND Flash, 모바일AP 등'},
    {'id': 'S05', 'company': '삼성전자', 'question': '삼성전자 전장 사업에 해당하는 하만의 주요 제품은?', 'quote': 'Harman | 디지털 콕핏(Digital Cockpit), 카오디오, 포터블 스피커 등'},
    {'id': 'R01', 'company': '리노공업', 'question': '리노공업은 납품 단가를 정할 때 고객 주문에서 무엇을 살피는가?', 'quote': '거래처별 공급수량 및 제품의 난이도에 따라 합리적으로 가격을 책정'},
    {'id': 'R02', 'company': '리노공업', 'question': '리노공업의 해외 거래처 관리 방식은 무엇인가?', 'quote': '지역별 혹은 국가별로 현지 대리점을 두어 고객을 관리하고 있으며'},
    {'id': 'R03', 'company': '리노공업', 'question': '리노공업의 공정가치로 측정되는 금융상품 관련 공시 근거를 찾아줘.', 'quote': '공정가치로 측정되는 금융상품에 대하여 공정가치 측정'},
    {'id': 'R04', 'company': '리노공업', 'question': '리노공업 퇴직급여 표에서 확정급여채무와 함께 어떤 적립자산 항목을 확인할 수 있나?', 'quote': '확정급여채무의 현재가치 | 사외적립자산'},
    {'id': 'R05', 'company': '리노공업', 'question': '리노공업이 시장 환경에서 경계하는 원자재·에너지 관련 요인은?', 'quote': '국내외 원자재 가격과 유가의 불안정'},
]
display(pd.DataFrame(CASES)[['id', 'company', 'question']])
print('질문 10개 | 기업별 5개 | 27번 질문과 문장 중복 없음')

,id,company,question
0,S01,삼성전자,삼성전자가 주식을 일반에 공개한 시점은 언제인가?
1,S02,삼성전자,삼성전자공업에서 지금의 회사명으로 바뀐 날짜는?
2,S03,삼성전자,삼성전자 본사의 소재지는 어디인가?
3,S04,삼성전자,삼성전자 반도체 사업 부문은 어떤 제품을 주로 만드는가?
4,S05,삼성전자,삼성전자 전장 사업에 해당하는 하만의 주요 제품은?
5,R01,리노공업,리노공업은 납품 단가를 정할 때 고객 주문에서 무엇을 살피는가?
6,R02,리노공업,리노공업의 해외 거래처 관리 방식은 무엇인가?
7,R03,리노공업,리노공업의 공정가치로 측정되는 금융상품 관련 공시 근거를 찾아줘.
8,R04,리노공업,리노공업 퇴직급여 표에서 확정급여채무와 함께 어떤 적립자산 항목을 확인할 수 있나?
9,R05,리노공업,리노공업이 시장 환경에서 경계하는 원자재·에너지 관련 요인은?


질문 10개 | 기업별 5개 | 27번 질문과 문장 중복 없음


## 2. 원문 준비와 E5 기준 청킹

27번에서는 LlamaIndex의 기본 토큰 기준으로 512를 맞췄지만, E5 토크나이저가 세는 길이는 달랐습니다. 여기서는 **E5 토크나이저 자체**를 `SentenceSplitter`에 전달하고 청크 목표를 400으로 낮춰 `passage: ` 접두어·특수 토큰의 여유를 둡니다. 이후 실제 E5 입력 길이를 **전수 검사**하고, 임베딩에서도 `truncation=False`로 잘림을 금지합니다.

In [3]:
def download_xml(rcept_no, xml_name):
    try:
        with requests.get('https://opendart.fss.or.kr/api/document.xml',
                          params={'crtfc_key': API_KEY, 'rcept_no': rcept_no},
                          timeout=120, stream=True) as response:
            if response.status_code != 200:
                raise RuntimeError(f'OpenDART 원문 HTTP {response.status_code}')
            parts, received = [], 0
            for part in response.iter_content(chunk_size=1024 * 1024):
                received += len(part)
                if received > 60_000_000:
                    raise RuntimeError('원문 ZIP이 60MB를 넘어 중단합니다.')
                parts.append(part)
    except requests.RequestException:
        raise RuntimeError('OpenDART 원문 요청 실패: 연결·인증·API 상태를 확인하세요. 요청 URL은 출력하지 않습니다.') from None
    payload = b''.join(parts)
    if not zipfile.is_zipfile(BytesIO(payload)):
        raise RuntimeError('OpenDART 응답이 ZIP 형식이 아닙니다. 키·접수번호·API 상태를 확인하세요.')
    try:
        with zipfile.ZipFile(BytesIO(payload)) as archive:
            if xml_name not in archive.namelist():
                raise RuntimeError('지정한 사업보고서 XML을 ZIP에서 찾지 못했습니다.')
            info = archive.getinfo(xml_name)
            if info.file_size > 60_000_000 or sum(item.file_size for item in archive.infolist()) > 120_000_000:
                raise RuntimeError('ZIP 압축 해제 크기가 실험 한도를 넘습니다.')
            return archive.read(xml_name)
    except zipfile.BadZipFile:
        raise RuntimeError('원문 ZIP이 손상됐습니다. API 응답 형식과 접수번호를 확인하세요.') from None

def extract_b_text(xml_bytes):
    soup = BeautifulSoup(xml_bytes, 'lxml-xml')
    if soup.find() is None:
        raise RuntimeError('XML에서 태그를 찾지 못했습니다.')
    parts = []
    for node in soup.descendants:
        if isinstance(node, Tag) and node.name.upper() == 'TR':
            parts.append('\n')
        if not isinstance(node, NavigableString) or isinstance(node, Comment):
            continue
        value = re.sub(r'\s+', ' ', str(node)).strip()
        if not value:
            continue
        ancestors = {tag.name.upper() for tag in node.parents if tag.name}
        if ancestors & {'TITLE', 'SUBTITLE'}:
            parts.append('\n## ' + value + '\n')
        elif ancestors & {'TD', 'TH'}:
            parts.append(value + ' | ')
        else:
            parts.append(value + ' ')
    return re.sub(r' *\n *', '\n', ''.join(parts)).strip()

In [4]:
MODEL_ID = 'intfloat/multilingual-e5-small'
MODEL_REVISION = '5697a65b0a002a92fe8c4fc9d495303ffff9c7d2'
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REVISION)

def e5_length(text):
    return len(tokenizer('passage: ' + text, add_special_tokens=True, truncation=False)['input_ids'])

old_splitter = SentenceSplitter(chunk_size=512, chunk_overlap=64)
safe_splitter = SentenceSplitter(chunk_size=400, chunk_overlap=48,
                                 tokenizer=lambda text: tokenizer.encode(text, add_special_tokens=False))
compact = lambda text: re.sub(r'\s+', '', text)
chunks_by_company, audit_rows = {}, []
for company, report in REPORTS.items():
    started = perf_counter()
    source_text = extract_b_text(download_xml(report['rcept_no'], report['xml_name']))
    sections = [part for part in re.split(r'(?=^## )', source_text, flags=re.MULTILINE) if part.strip()]
    old_chunks = [node.text.strip() for section in sections
                  for node in old_splitter.get_nodes_from_documents([Document(text=section)]) if node.text.strip()]
    chunks = [node.text.strip() for section in sections
              for node in safe_splitter.get_nodes_from_documents([Document(text=section)]) if node.text.strip()]
    chunks_by_company[company] = chunks
    lengths = [e5_length(chunk) for chunk in chunks]
    audit_rows.append({'company': company, 'old_chunks': len(old_chunks),
                       'old_over_512': sum(e5_length(chunk) > 512 for chunk in old_chunks),
                       'new_chunks': len(chunks), 'new_over_512': sum(length > 512 for length in lengths),
                       'new_max_tokens': max(lengths),
                       'prepare_seconds': round(perf_counter() - started, 2)})
audit = pd.DataFrame(audit_rows)
display(audit)
if audit['new_over_512'].sum() != 0:
    raise RuntimeError('E5 512-token 초과 청크가 남아 있습니다. 임베딩을 중단합니다.')
missing = [case['id'] for case in CASES if not any(compact(case['quote']) in compact(chunk)
           for chunk in chunks_by_company[case['company']])]
if missing:
    raise RuntimeError(f'정답 근거가 새 청크에서 누락된 질문: {missing}. 문항 또는 청킹을 확인한 뒤 계속하세요.')
print('길이·근거 검사 완료 | 초과 청크 0개 | 질문 10개 모두 근거 청크 존재')

Token indices sequence length is longer than the specified maximum sequence length for this model (2734 > 512). Running this sequence through the model will result in indexing errors


,company,old_chunks,old_over_512,new_chunks,new_over_512,new_max_tokens,prepare_seconds
0,삼성전자,96,0,82,0,403,0.77
1,리노공업,384,2,333,0,404,1.75


길이·근거 검사 완료 | 초과 청크 0개 | 질문 10개 모두 근거 청크 존재


## 3. 공통 청크에 문자·BM25·E5 인덱스 만들기

문자 검색은 글자 2~4-gram의 유사도, BM25는 Kiwi 형태소로 토큰화한 뒤 SQLite FTS5의 순위입니다. 두 방식 모두 **같은 청크**를 인덱싱합니다. BM25는 일치하는 토큰이 없는 청크를 후보에 넣지 않습니다.

In [5]:
vectorizer = HashingVectorizer(analyzer='char', ngram_range=(2, 4),
                               n_features=2**18, norm='l2', alternate_sign=False, dtype=np.float32)
kiwi = Kiwi()

def morph_terms(text):
    terms = []
    for token in kiwi.tokenize(text):
        if token.tag.startswith(('NN', 'VV', 'VA')) or token.tag in {'SL', 'SN', 'XR'}:
            terms.extend(re.findall(r'[^\W_]+', token.form.casefold()))
    return terms

char_index, bm25_connections, build_rows = {}, {}, []
for company, chunks in chunks_by_company.items():
    started = perf_counter()
    char_index[company] = vectorizer.fit_transform(chunks)
    char_seconds = perf_counter() - started
    started = perf_counter()
    tokenized = [' '.join(morph_terms(chunk)) for chunk in chunks]
    connection = sqlite3.connect(':memory:')
    connection.execute("CREATE VIRTUAL TABLE passage USING fts5(terms, content='')")
    connection.executemany('INSERT INTO passage(rowid, terms) VALUES (?, ?)',
                           [(index + 1, terms) for index, terms in enumerate(tokenized)])
    connection.commit()
    bm25_connections[company] = connection
    page_count = connection.execute('PRAGMA page_count').fetchone()[0]
    page_size = connection.execute('PRAGMA page_size').fetchone()[0]
    build_rows.append({'company': company, 'char_index_seconds': round(char_seconds, 2),
                       'bm25_tokenize_index_seconds': round(perf_counter() - started, 2),
                       'empty_bm25_chunks': sum(not terms for terms in tokenized),
                       'bm25_sqlite_MB': round(page_count * page_size / 1024**2, 2)})
display(pd.DataFrame(build_rows))

,company,char_index_seconds,bm25_tokenize_index_seconds,empty_bm25_chunks,bm25_sqlite_MB
0,삼성전자,0.04,1.76,0,0.06
1,리노공업,0.15,1.51,0,0.14


In [6]:
torch.set_num_threads(min(4, os.cpu_count() or 1))
print('E5 로드 시작 | 최초 실행이라면 모델 다운로드가 필요할 수 있습니다.')
started = perf_counter()
model = AutoModel.from_pretrained(MODEL_ID, revision=MODEL_REVISION, use_safetensors=True).to('cpu').eval()
model_load_seconds = perf_counter() - started
model_parameter_MB = sum(p.numel() * p.element_size() for p in model.parameters()) / 1024**2
print(f'E5 로드 완료 | {model_load_seconds:.1f}초 | 파라미터 약 {model_parameter_MB:.0f} MiB')

def encode_texts(texts, kind, batch_size=16):
    rows = []
    for start in range(0, len(texts), batch_size):
        batch = [f'{kind}: {text}' for text in texts[start:start + batch_size]]
        tokens = tokenizer(batch, padding=True, truncation=False, return_tensors='pt')
        if tokens['input_ids'].shape[1] > 512:
            raise RuntimeError('E5 입력 길이가 512를 넘었습니다. 잘라내지 않고 중단합니다.')
        with torch.inference_mode():
            hidden = model(**tokens).last_hidden_state
            mask = tokens['attention_mask'].unsqueeze(-1)
            pooled = (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
            rows.append(F.normalize(pooled, p=2, dim=1).cpu().numpy().astype(np.float32))
    return np.vstack(rows)

dense_index, dense_rows = {}, []
for company, chunks in chunks_by_company.items():
    print(f'{company} E5 passage 임베딩 시작 | {len(chunks)}개')
    started = perf_counter()
    dense_index[company] = encode_texts(chunks, 'passage')
    dense_rows.append({'company': company, 'dense_index_seconds': round(perf_counter() - started, 2),
                       'dense_MB': round(dense_index[company].nbytes / 1024**2, 2)})
display(pd.DataFrame(dense_rows))

E5 로드 시작 | 최초 실행이라면 모델 다운로드가 필요할 수 있습니다.
E5 로드 완료 | 0.2초 | 파라미터 약 449 MiB
삼성전자 E5 passage 임베딩 시작 | 82개
리노공업 E5 passage 임베딩 시작 | 333개


,company,dense_index_seconds,dense_MB
0,삼성전자,4.9,0.12
1,리노공업,17.0,0.49


## 4. 같은 질문에서 두 하이브리드 검색 실행

각 질문의 E5 검색 결과를 **한 번만 계산**해 두 하이브리드에 공유합니다. 점수 단위가 다른 검색기를 직접 더하지 않고 RRF로 순위를 결합합니다. 단독 문자·BM25·E5 결과도 실패 원인 파악용으로 함께 기록합니다.

In [7]:
def bm25_order(company, question):
    terms = list(dict.fromkeys(morph_terms(question)))
    if not terms:
        return []
    match_query = ' OR '.join(f'"{term}"' for term in terms)
    rows = bm25_connections[company].execute(
        'SELECT rowid FROM passage WHERE passage MATCH ? ORDER BY rank, rowid', (match_query,)
    ).fetchall()
    return [rowid - 1 for (rowid,) in rows]

def rrf_order(lex_order, dense_order, size, depth=20, k=60):
    scores = np.zeros(size, dtype=np.float64)
    for order in (lex_order, dense_order):
        ids = np.asarray(order[:depth], dtype=int)
        scores[ids] += 1.0 / (k + np.arange(1, len(ids) + 1))
    order = np.argsort(-scores, kind='stable')
    return order[scores[order] > 0]

def evidence_rank(order, chunks, quote):
    needle = compact(quote)
    return next((rank for rank, index in enumerate(order, 1) if needle in compact(chunks[index])), None)

assert list(rrf_order([0, 1, 2], [0, 2, 1], 3)) == [0, 1, 2]
assert evidence_rank([1, 0], ['정답 문장', '오답'], '정답문장') == 2
print('RRF·근거 순위 최소 검사 완료')

RRF·근거 순위 최소 검사 완료


In [8]:
rows = []
for case in CASES:
    company, question = case['company'], case['question']
    chunks = chunks_by_company[company]
    started = perf_counter()
    char_scores = (char_index[company] @ vectorizer.transform([question]).T).toarray().ravel()
    char_order = np.argsort(-char_scores, kind='stable')
    char_ms = (perf_counter() - started) * 1000
    started = perf_counter()
    bm_order = bm25_order(company, question)
    bm_ms = (perf_counter() - started) * 1000
    started = perf_counter()
    dense_query = encode_texts([question], 'query')[0]
    dense_order = np.argsort(-(dense_index[company] @ dense_query), kind='stable')
    dense_ms = (perf_counter() - started) * 1000
    started = perf_counter()
    char_hybrid = rrf_order(char_order, dense_order, len(chunks))
    char_fusion_ms = (perf_counter() - started) * 1000
    started = perf_counter()
    bm_hybrid = rrf_order(bm_order, dense_order, len(chunks))
    bm_fusion_ms = (perf_counter() - started) * 1000
    methods = [
        ('문자', char_order, char_ms), ('BM25', bm_order, bm_ms), ('E5', dense_order, dense_ms),
        ('문자+E5', char_hybrid, char_ms + dense_ms + char_fusion_ms),
        ('BM25+E5', bm_hybrid, bm_ms + dense_ms + bm_fusion_ms),
    ]
    for method, order, latency in methods:
        rank = evidence_rank(order, chunks, case['quote'])
        rows.append({'id': case['id'], 'company': company, 'method': method, 'rank': rank,
                     'ranked_count': len(order), 'bm25_match_count': len(bm_order),
                     'hit_at_3': rank is not None and rank <= 3,
                     'hit_at_5': rank is not None and rank <= 5,
                     'rr_at_5': 1 / rank if rank is not None and rank <= 5 else 0.0,
                     'query_ms': round(latency, 2),
                     'top1_excerpt': chunks[order[0]][:160] if len(order) else ''})
    print(f"[{case['id']}] 문자+E5 {rows[-2]['rank']}위 | BM25+E5 {rows[-1]['rank']}위")
results = pd.DataFrame(rows)
display(results[['id', 'company', 'method', 'rank', 'ranked_count', 'bm25_match_count', 'hit_at_3', 'hit_at_5']])

[S01] 문자+E5 1위 | BM25+E5 2위
[S02] 문자+E5 1위 | BM25+E5 1위
[S03] 문자+E5 1위 | BM25+E5 1위
[S04] 문자+E5 3위 | BM25+E5 3위
[S05] 문자+E5 4위 | BM25+E5 4위
[R01] 문자+E5 None위 | BM25+E5 None위
[R02] 문자+E5 4위 | BM25+E5 1위
[R03] 문자+E5 10위 | BM25+E5 3위
[R04] 문자+E5 3위 | BM25+E5 4위
[R05] 문자+E5 3위 | BM25+E5 6위


,id,company,method,rank,ranked_count,bm25_match_count,hit_at_3,hit_at_5
0,S01,삼성전자,문자,1.0,82,27,True,True
1,S01,삼성전자,BM25,4.0,27,27,False,True
2,S01,삼성전자,E5,2.0,82,27,True,True
3,S01,삼성전자,문자+E5,1.0,32,27,True,True
4,S01,삼성전자,BM25+E5,2.0,29,27,True,True
5,S02,삼성전자,문자,1.0,82,8,True,True
6,S02,삼성전자,BM25,1.0,8,8,True,True
7,S02,삼성전자,E5,3.0,82,8,True,True
8,S02,삼성전자,문자+E5,1.0,32,8,True,True
9,S02,삼성전자,BM25+E5,1.0,24,8,True,True


In [9]:
summary = results.groupby('method', as_index=False).agg(
    cases=('id', 'count'), hit_at_3=('hit_at_3', 'mean'), hit_at_5=('hit_at_5', 'mean'),
    mrr_at_5=('rr_at_5', 'mean'), median_query_ms=('query_ms', 'median')
)
display(summary)
paired = results[results['method'] == '문자+E5'][['id', 'company', 'rank', 'hit_at_5']].rename(
    columns={'rank': '문자+E5_순위', 'hit_at_5': '문자+E5_Hit5'}
).merge(results[results['method'] == 'BM25+E5'][['id', 'rank', 'hit_at_5']].rename(
    columns={'rank': 'BM25+E5_순위', 'hit_at_5': 'BM25+E5_Hit5'}), on='id')
display(paired)
print('두 하이브리드가 함께 놓쳤거나 Hit@5 판정이 다른 질문')
display(paired[(~paired['문자+E5_Hit5']) | (~paired['BM25+E5_Hit5'])])
print('실패 근거 확인용 상위 1개 청크 발췌')
display(results[(results['method'].isin(['문자+E5', 'BM25+E5'])) & (~results['hit_at_5'])]
        [['id', 'method', 'rank', 'ranked_count', 'bm25_match_count', 'top1_excerpt']])

,method,cases,hit_at_3,hit_at_5,mrr_at_5,median_query_ms
0,BM25,10,0.3,0.6,0.320000,1.275
1,BM25+E5,10,0.6,0.8,0.466667,17.655
2,E5,10,0.7,0.9,0.445000,16.145
3,문자,10,0.4,0.4,0.350000,2.225
4,문자+E5,10,0.6,0.8,0.450000,18.425


,id,company,문자+E5_순위,문자+E5_Hit5,BM25+E5_순위,BM25+E5_Hit5
0,S01,삼성전자,1.0,True,2.0,True
1,S02,삼성전자,1.0,True,1.0,True
2,S03,삼성전자,1.0,True,1.0,True
3,S04,삼성전자,3.0,True,3.0,True
4,S05,삼성전자,4.0,True,4.0,True
5,R01,리노공업,NaN,False,NaN,False
6,R02,리노공업,4.0,True,1.0,True
7,R03,리노공업,10.0,False,3.0,True
8,R04,리노공업,3.0,True,4.0,True
9,R05,리노공업,3.0,True,6.0,False


두 하이브리드가 함께 놓쳤거나 Hit@5 판정이 다른 질문


,id,company,문자+E5_순위,문자+E5_Hit5,BM25+E5_순위,BM25+E5_Hit5
5,R01,리노공업,NaN,False,NaN,False
7,R03,리노공업,10.0,False,3.0,True
9,R05,리노공업,3.0,True,6.0,False


실패 근거 확인용 상위 1개 청크 발췌


,id,method,rank,ranked_count,bm25_match_count,top1_excerpt
28,R01,문자+E5,NaN,32,61,이에 상응하는 기술력 확보 |\n연구과제 | TSV용 프로브카드의 개발 |\n연구기...
29,R01,BM25+E5,NaN,32,61,"아울러 반도체 시장의 변화에 따른 원가 절감 및 성능 향상을 위한 공정개선, 즉 반..."
38,R03,문자+E5,10.0,32,225,"| 금융자산은 최초인식시 공정가치로 측정하고 있으며, 당기손익-공정가치금융자산이아닌..."
49,R05,BM25+E5,6.0,29,122,"|\n상품화 내용 - 시제품 검증을 마치고 유효성이 확인되었으며, 고사양 High ..."


## 결과를 읽는 기준

1. 길이 감사에서 `new_over_512=0`이고 정답 인용문 10개가 청크에 있어야 검색 결과를 해석할 수 있습니다. 27번의 초과 청크를 **버리는 것이 아니라 다시 나누는 것**입니다.
2. 주 비교는 `문자+E5` 대 `BM25+E5`입니다. 문자·BM25·E5 단독 결과는 원인 분석용입니다. 회사별 5개, 전체 10개라 작은 차이는 우연일 수 있습니다. 질문 작성자가 기존 보고서 근거를 알고 있었으므로 독립 holdout은 아닙니다.
3. `BM25+E5`에서 근거가 사라졌다면 형태소 불일치, BM25의 후보 누락, RRF 상위 20개 제한을 구분해 봅니다. `bm25_match_count`는 BM25 MATCH 결과 수이고 `ranked_count`는 해당 방식의 순위 길이입니다. 하이브리드의 `ranked_count`는 RRF에 들어온 두 순위의 상위 20개 합집합 크기입니다.
4. 속도는 CPU 로컬 warm 질의 측정입니다. 두 하이브리드 시간에 **공통 E5 질문 임베딩 시간**을 각각 포함했으며, 최초 모델 로드·청크 임베딩·인덱스 구축은 별도로 기록했습니다. 로컬 속도만으로 채택하지 않습니다.
5. 서로 다른 지표가 엇갈리거나 근거가 없는 질문이 발견되면 선택을 보류하고 질문·근거를 먼저 수정합니다. 이번 결과만으로 최종 운영 검색기를 확정하지 않습니다.

참고: [multilingual-E5-small 모델 카드](https://huggingface.co/intfloat/multilingual-e5-small), [LlamaIndex SentenceSplitter](https://developers.llamaindex.ai/python/framework-api-reference/node_parsers/sentence_splitter/), [SQLite FTS5 BM25](https://www.sqlite.org/fts5.html#the_bm25_function), [Kiwi](https://bab2min.github.io/kiwipiepy/).

## 5. 같은 청크에서 E5와 OpenAI 임베딩 비교

이 절은 1~4절을 실행한 **같은 커널**에서 이어서 실행합니다. E5와 OpenAI `text-embedding-3-small`은 모두 벡터를 만드는 모델입니다. OpenAI API에는 E5에서 사용한 **동일한 청크 원문·질문 원문**을 보내며, 문자/BM25 순위·RRF(상위 20, k=60)·정답 근거·지표는 그대로 둡니다. OpenAI에는 E5 전용 `query:`/`passage:` 접두어를 붙이지 않습니다.

**주의:** 아래 API 셀은 공개 사업보고서 청크를 OpenAI에 전송하고 입력 토큰에 따른 비용이 발생합니다. 재실행하면 다시 호출·과금됩니다. `.env`에 `OPENAI_API_KEY`가 있어야 합니다. 키와 요청 본문 전체는 출력하지 않습니다. API 지연시간과 로컬 E5 지연시간은 실행 환경이 달라 품질 우열 판단에 사용하지 않습니다.

In [10]:
from openai import OpenAI, OpenAIError

OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
if not OPENAI_API_KEY:
    raise RuntimeError('.env에 OPENAI_API_KEY가 필요합니다. 키 값은 출력하지 않습니다.')
api_client = OpenAI(api_key=OPENAI_API_KEY, timeout=120, max_retries=2)
OPENAI_EMBED_MODEL = 'text-embedding-3-small'

def api_embed(texts, label, batch_size=32):
    matrices, used_tokens, calls = [], 0, 0
    started = perf_counter()
    for start in range(0, len(texts), batch_size):
        batch = texts[start:start + batch_size]
        print(f'{label} {start + 1}~{start + len(batch)}/{len(texts)} 호출 시작', flush=True)
        try:
            response = api_client.embeddings.create(model=OPENAI_EMBED_MODEL, input=batch, encoding_format='float')
        except OpenAIError as exc:
            raise RuntimeError(f'OpenAI 임베딩 요청 실패: {type(exc).__name__}. 키·한도·연결을 확인하세요.') from None
        if response.model != OPENAI_EMBED_MODEL:
            raise RuntimeError('요청한 모델과 다른 임베딩 모델이 응답했습니다.')
        ordered = sorted(response.data, key=lambda item: item.index)
        if [item.index for item in ordered] != list(range(len(batch))):
            raise RuntimeError('OpenAI 임베딩 응답의 입력 순서·개수가 다릅니다.')
        matrix = np.asarray([item.embedding for item in ordered], dtype=np.float32)
        norms = np.linalg.norm(matrix, axis=1, keepdims=True)
        if not np.isfinite(matrix).all() or not np.isfinite(norms).all() or (norms == 0).any():
            raise RuntimeError('OpenAI 임베딩에 비정상 벡터가 포함됐습니다.')
        if calls == 0:
            print(f'응답 모델: {response.model} | 차원: {matrix.shape[1]}')
        matrices.append(matrix / norms)
        used_tokens += response.usage.total_tokens
        calls += 1
    return np.vstack(matrices), used_tokens, calls, perf_counter() - started

print('OpenAI 임베딩 비교 준비 완료 | 키 값은 출력하지 않음')

OpenAI 임베딩 비교 준비 완료 | 키 값은 출력하지 않음


In [11]:
openai_dense_index, passage_usage = {}, []
for company, chunks in chunks_by_company.items():
    matrix, tokens, calls, seconds = api_embed(chunks, f'{company} passage')
    if matrix.shape != (len(chunks), 1536):
        raise RuntimeError('OpenAI 임베딩 차원 또는 청크 개수가 예상과 다릅니다.')
    openai_dense_index[company] = matrix
    passage_usage.append({'company': company, 'chunks': len(chunks), 'api_calls': calls,
                          'input_tokens': tokens, 'api_seconds': round(seconds, 2),
                          'vector_MB': round(matrix.nbytes / 1024**2, 2)})
display(pd.DataFrame(passage_usage))
print('주의: 이 셀을 다시 실행하면 모든 청크를 다시 전송·과금합니다.')

삼성전자 passage 1~32/82 호출 시작
응답 모델: text-embedding-3-small | 차원: 1536
삼성전자 passage 33~64/82 호출 시작
삼성전자 passage 65~82/82 호출 시작
리노공업 passage 1~32/333 호출 시작
응답 모델: text-embedding-3-small | 차원: 1536
리노공업 passage 33~64/333 호출 시작
리노공업 passage 65~96/333 호출 시작
리노공업 passage 97~128/333 호출 시작
리노공업 passage 129~160/333 호출 시작
리노공업 passage 161~192/333 호출 시작
리노공업 passage 193~224/333 호출 시작
리노공업 passage 225~256/333 호출 시작
리노공업 passage 257~288/333 호출 시작
리노공업 passage 289~320/333 호출 시작
리노공업 passage 321~333/333 호출 시작


,company,chunks,api_calls,input_tokens,api_seconds,vector_MB
0,삼성전자,82,3,38110,2.73,0.48
1,리노공업,333,11,142960,7.40,1.95


주의: 이 셀을 다시 실행하면 모든 청크를 다시 전송·과금합니다.


In [12]:
questions = [case['question'] for case in CASES]
openai_query_matrix, query_tokens, query_calls, query_seconds = api_embed(questions, '질문')
if openai_query_matrix.shape != (len(CASES), 1536):
    raise RuntimeError('OpenAI 질문 임베딩 개수·차원이 예상과 다릅니다.')
total_tokens = sum(row['input_tokens'] for row in passage_usage) + query_tokens
print(f'API 사용량 | 총 {total_tokens:,} 입력 토큰 | {sum(row["api_calls"] for row in passage_usage) + query_calls}회 호출')
print(f'질문 {len(CASES)}개 묶음 임베딩 시간: {query_seconds:.2f}초 (질문당 지연시간 아님)')
print(f'참고 비용 약 ${total_tokens / 1_000_000 * 0.02:.4f} | 2026-10-05 공식 단가 $0.02/1M 입력 토큰 기준')
print('성공 응답 usage 기준 추정치이며, 재시도와 실제 청구액은 다를 수 있습니다.')

질문 1~10/10 호출 시작
응답 모델: text-embedding-3-small | 차원: 1536
API 사용량 | 총 181,397 입력 토큰 | 15회 호출
질문 10개 묶음 임베딩 시간: 0.29초 (질문당 지연시간 아님)
참고 비용 약 $0.0036 | 2026-10-05 공식 단가 $0.02/1M 입력 토큰 기준
성공 응답 usage 기준 추정치이며, 재시도와 실제 청구액은 다를 수 있습니다.


In [13]:
api_rows = []
for case, query_vector in zip(CASES, openai_query_matrix):
    company, question = case['company'], case['question']
    chunks = chunks_by_company[company]
    char_scores = (char_index[company] @ vectorizer.transform([question]).T).toarray().ravel()
    char_order = np.argsort(-char_scores, kind='stable')
    bm_order = bm25_order(company, question)
    api_order = np.argsort(-(openai_dense_index[company] @ query_vector), kind='stable')
    methods = [
        ('OpenAI', api_order),
        ('문자+OpenAI', rrf_order(char_order, api_order, len(chunks))),
        ('BM25+OpenAI', rrf_order(bm_order, api_order, len(chunks))),
    ]
    for method, order in methods:
        rank = evidence_rank(order, chunks, case['quote'])
        api_rows.append({'id': case['id'], 'company': company, 'method': method, 'rank': rank,
                         'hit_at_3': rank is not None and rank <= 3,
                         'hit_at_5': rank is not None and rank <= 5,
                         'rr_at_5': 1 / rank if rank is not None and rank <= 5 else 0.0})
api_results = pd.DataFrame(api_rows)
quality = pd.concat([results[['id', 'company', 'method', 'rank', 'hit_at_3', 'hit_at_5', 'rr_at_5']],
                     api_results], ignore_index=True)
display(quality.groupby('method', as_index=False).agg(
    cases=('id', 'count'), hit_at_3=('hit_at_3', 'mean'),
    hit_at_5=('hit_at_5', 'mean'), mrr_at_5=('rr_at_5', 'mean')))
display(quality.pivot(index='id', columns='method', values='rank')
        [['E5', 'OpenAI', '문자+E5', '문자+OpenAI', 'BM25+E5', 'BM25+OpenAI']])

,method,cases,hit_at_3,hit_at_5,mrr_at_5
0,BM25,10,0.3,0.6,0.320000
1,BM25+E5,10,0.6,0.8,0.466667
2,BM25+OpenAI,10,0.6,0.6,0.383333
3,E5,10,0.7,0.9,0.445000
4,OpenAI,10,0.5,0.7,0.500000
5,문자,10,0.4,0.4,0.350000
6,문자+E5,10,0.6,0.8,0.450000
7,문자+OpenAI,10,0.5,0.6,0.408333


method,E5,OpenAI,문자+E5,문자+OpenAI,BM25+E5,BM25+OpenAI
id,,,,,,
R01,73.0,9.0,NaN,14.0,NaN,16.0
R02,3.0,39.0,4.0,18.0,1.0,8.0
R03,1.0,4.0,10.0,12.0,3.0,6.0
R04,5.0,2.0,3.0,2.0,4.0,2.0
R05,4.0,22.0,3.0,18.0,6.0,25.0
S01,2.0,4.0,1.0,1.0,2.0,3.0
S02,3.0,1.0,1.0,1.0,1.0,1.0
S03,2.0,1.0,1.0,1.0,1.0,1.0
S04,3.0,1.0,3.0,4.0,3.0,2.0


### 모델 비교 해석

- `E5` 대 `OpenAI`, `문자+E5` 대 `문자+OpenAI`, `BM25+E5` 대 `BM25+OpenAI`는 각각 **검색 구조를 고정하고 임베딩 모델만** 바꾼 쌍입니다. 다른 차원의 벡터끼리 직접 비교하거나 혼합하지 않습니다.
- OpenAI 입력은 E5가 허용하는 청크 길이로 맞췄습니다. 따라서 **현재 공통 청크에서의 모델 비교**이지, OpenAI의 긴 입력 한도까지 활용한 최적 구성 비교는 아닙니다. 질문셋 10개는 개발용이므로 최종 모델 선택에는 독립 질문으로 재검증이 필요합니다.
- `api_seconds`는 통신·서버 처리 등을 포함한 묶음 호출 시간이고, 4절의 `median_query_ms`는 로컬 CPU 단건 측정입니다. 속도를 직접 우열 비교하지 않습니다. 토큰 비용은 실행 시점의 [공식 가격](https://developers.openai.com/api/docs/models/text-embedding-3-small)을 다시 확인하세요.

참고: [OpenAI Docs: 임베딩](https://developers.openai.com/api/docs/guides/embeddings), [OpenAI API: 다중 입력·사용량 응답](https://developers.openai.com/api/reference/resources/embeddings/methods/create), [E5 모델 카드](https://huggingface.co/intfloat/multilingual-e5-small/raw/main/README.md).